# 03 - PHASE 0 GATE: does the efficiency structure live at the CLASS level?

AGENTS.md Sec 5. The evidence that motivated this project came from synthetic
inject-then-recover, which is circular. It must replicate on REAL logits before Phase 1.

Three mechanisms, all using the SAME conformal offset estimator on abundant data, differing only
in what INDEXES the correction:

1. a single global **temperature** (1 parameter),
2. a per-sample offset indexed by **free energy** `E(x) = -logsumexp(logits)` (n_bins),
3. a per-**class** offset (K parameters).

## Metric: Amendment 6 (the Amendment 3 set-size criterion was RETIRED)

The previous criterion - average set size at worst-class coverage - **cannot answer Sec 5**, and
the first Pl@ntNet run was scored with it. Three independent legs:

1. **Nested capacity degraded monotonically** on the real logits: `energy_b2 -47.2`,
   `energy_b10 -67.4`, `energy_b50 -74.7`. b50 can represent everything b2 can, so a criterion
   where extra capacity always loses is not measuring where structure lives.
2. **Zero discriminating power.** On four synthetic worlds each carrying exactly one planted
   mechanism it named `temperature` the winner in **4 of 4** - including the world whose only
   structure was per-class difficulty.
3. **The arithmetic.** Reaching worst-class coverage by UNIFORM inflation charges a mechanism for
   its threshold VARIANCE whatever the source. The classes still short after class-adaptation are
   the ones with the LOWEST thresholds, so a global additive repair over-inflates the whole
   vector; and with `s = 1 - p` on a weak model nearly every wrong label sits in [0.98, 1.0], so
   **+0.0117 of inflation took average set size from 13.6 to 55.7**.

Set size conflates Sec 5 with score saturation, the repair operator, and the
conditional-vs-marginal tradeoff (equalizing coverage across classes *always* costs average set
size - that is Jensen, not a finding). No patch removes those, so the question was changed.

### What is measured instead

Target `q*_y` = the (1-alpha) quantile of class y true-label scores on EVAL. Each mechanism is fit
on CAL and predicts it; **R2 is taken across CLASSES, out of sample**:

| mechanism | class-level prediction |
|---|---|
| `global` | a constant (no class-level variance by construction) |
| `energy_bK` | class-mean of the per-sample thresholds it assigns |
| `class` | the per-class q_hat fit on CAL |

Noise parameters earn **negative** R2 (measured `energy_b50 -> -35.8` in the no-structure world),
so capacity is punished, not rewarded. And the rival is not crippled: when class difficulty is
real it also shows up in per-sample confidence, so `energy_b10` reached **+0.523** against
`class`'s **+0.969** in the planted-class world.

**Temperature is judged by the question it can actually answer.** A global temperature cannot
produce class-level variance, so scoring it by class-level R2 would rig the comparison. Instead:
does *any* global temperature REMOVE the class-level structure? The verdict uses the **worst**
reliability over a temperature scan, not just the efficiency-optimal T.

**PASS (Sec 5), pre-registered:** structure exists above noise (reliability > 0.30) AND survives
every temperature AND `class` R2 beats every energy rival with non-overlapping CIs.

Reported alongside: `sd(q*_y)`, the Sec 5 discriminant - **0.0839** with planted class structure
vs **0.0020** with only global miscalibration.

**This verdict is readable only because the metric was validated 4/4 on planted worlds first**
(`pcc/tests/test_phase0_explain.py`). The retired criterion scored 0/1 on the planted-class world.

## Sec 5 demands ABUNDANT data - which on Pl@ntNet means the HEAD

Sec 5 says the per-class offset is fit on *data berlimpah (bukan pada budget realistis)*. Pl@ntNet's
cal split has a **median of 2 samples per class** and 105 classes with none, so on the full label
space the class mechanism would be measuring estimation noise, exactly as it did on CIFAR-100
(~50 cal/class gave +3.56 where 1000/class gave +40.89).

So the PRIMARY run restricts the whole problem to classes with at least `MIN_CAL_PER_CLASS`
calibration samples - the abundant-data regime the section asks for. Restriction is applied to
samples AND score columns (`restrict_to_classes`), so there is no measured-vs-unmeasured asymmetry
to exploit. The unrestricted run is reported as a SECONDARY view, and the two are never merged.

**Scores are OURS**, not LTC's released scores - the checkpoint gate did not pass
(reports/phase0_checkpoint_gate.md). Every conclusion here inherits that.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'
BACKBONE   = 'resnet50_ltc'
CAL_SPLIT  = 'cal'          # calibration/eval pool (OUR scores)

ALPHAS     = (0.05, 0.1)    # alpha=0.01 needs n>=99 per class: only 57/1081 classes
MIN_CAL_PER_CLASS = 50      # PRIMARY: the abundant-data regime Sec 5 asks for
N_SPLITS   = 50             # random cal/eval splits (Sec 8.4 asks >=100; 50 keeps runtime sane,
                            # raise if the CIs are too wide to decide)
BIN_GRID   = (2, 10, 50)
RUN_UNRESTRICTED = True     # SECONDARY view over all classes, reported separately
MIN_EVAL_PER_CLASS = 20     # eval samples needed for a meaningful per-class quantile
RELIABILITY_SPLITS = 30     # split-half reps inside each Phase-0 split
T_SCAN_SPLITS = 10          # reps for the temperature scan (reliability is flat in T)
RUN_COST_REPORT = True      # retired set-size criterion, kept as a COST report only
N_SPLITS_COST = 20          # it is the slow one and no verdict rests on it
N_SPLITS_SECONDARY = 10     # the all-1081-class view costs ~12 s/split (vs 0.3 s for the
                            # abundant subset), so 50 splits would be ~20 min for a view that
                            # is expected to be noise-dominated anyway. Fewer splits => WIDER
                            # CIs there; that is acceptable because no verdict rests on it.
SEED = 42
EMB_ROOT = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_ROOT =', EMB_ROOT, '| alphas', ALPHAS, '| min cal/class', MIN_CAL_PER_CLASS)


## 2. Mount Drive + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load OUR scores + report the class-count regime


In [ ]:
import numpy as np
from pcc.data.load import load_split, split_provenance, per_class_counts
from pcc.data.ltc_datasets import NUM_CLASSES

prov = split_provenance(EMB_ROOT, CAL_SPLIT)
print('scores_source:', prov.get('scores_source'))
print('under_gate_exception:', prov.get('under_gate_exception'))
d = load_split(EMB_ROOT, CAL_SPLIT, keys=['logits','labels'])
logits = np.asarray(d['logits'], np.float64)
labels = np.asarray(d['labels']).astype(int)
K = NUM_CLASSES[DATASET]
n = len(labels)
counts = per_class_counts(labels, K)
print(f'{CAL_SPLIT}: {n} samples, {K} classes, accuracy '
      f'{float((logits.argmax(1)==labels).mean()):.4f}')
nz = counts[counts>0]
print('cal samples/class percentiles:',
      {f'p{q}': int(np.percentile(nz,q)) for q in (0,25,50,75,90,100)})
ABUNDANT = np.where(counts >= MIN_CAL_PER_CLASS)[0]
print(f'ABUNDANT classes (>= {MIN_CAL_PER_CLASS} cal samples): {len(ABUNDANT)}/{K}'
      f'  covering {counts[ABUNDANT].sum()}/{n} samples'
      f' ({100*counts[ABUNDANT].sum()/n:.1f}%)')
assert len(ABUNDANT) >= 10, 'too few abundant classes to run the primary analysis'


## 4. PRIMARY (Amendment 6) - which mechanism EXPLAINS the class-level quantile?

The set-size criterion of Amendment 3 was **retired**: it named `temperature` the winner in
4 of 4 synthetic worlds, including the world whose only planted structure was per-class
difficulty. Reaching worst-class coverage by uniform inflation charges a mechanism for its
threshold VARIANCE whatever its source, so nested capacity degraded monotonically
(energy_b2 -47.2 -> b10 -67.4 -> b50 -74.7). See protocol_amendments.md#amendment-6.

Target `q*_y` = the (1-alpha) quantile of class y true-label scores on EVAL. Every mechanism
is fit on CAL and predicts it; R2 is taken ACROSS CLASSES, out of sample, so noise parameters
earn NEGATIVE R2. Temperature is judged by the only question it can answer: does it REMOVE
the class-level structure (split-half reliability of q_y on rescaled scores)?


In [ ]:
from pcc.eval import decomposition as dc
from pcc.eval.conformal import restrict_to_classes
from pcc.eval.stats import mean_ci
import numpy as np

MECHS = [f'energy_b{b}' for b in BIN_GRID] + ['class']
KEYS  = ['global'] + MECHS

def run_explain(class_subset, tag, n_splits=None):
    n_splits = N_SPLITS if n_splits is None else n_splits
    lg, lab, Ksub, _ = restrict_to_classes(logits, labels, class_subset)
    out = {}
    for alpha in ALPHAS:
        acc = {m: [] for m in KEYS}
        rel_id, rel_T, tsd, nsc, bT, bnd = [], [], [], [], [], []
        rng = np.random.default_rng(SEED)
        for _ in range(n_splits):
            idx = rng.permutation(len(lab)); cal, ev = idx[:len(lab)//2], idx[len(lab)//2:]
            r = dc.phase0_explain_class_level(lg, lab, Ksub, alpha, cal, ev,
                                             bin_grid=BIN_GRID, min_eval=MIN_EVAL_PER_CLASS,
                                             reliability_splits=RELIABILITY_SPLITS,
                                             T_scan_splits=T_SCAN_SPLITS)
            for m in KEYS: acc[m].append(r['r2'][m])
            rel_id.append(r['reliability_identity'])
            rel_T.append(r['reliability_min_over_T'])
            tsd.append(r['target_sd']); nsc.append(r['n_classes_scored'])
            bT.append(r['best_T']); bnd.append(r['best_T_at_boundary'])
        o = {m: mean_ci(np.array(acc[m], float)) for m in KEYS}
        o['_reliability_identity'] = mean_ci(np.array(rel_id, float))
        o['_reliability_after_T']  = mean_ci(np.array(rel_T, float))
        o['_target_sd'] = mean_ci(np.array(tsd, float))
        o['_n_classes_scored'] = float(np.mean(nsc))
        o['_best_T_mean'] = float(np.mean(bT))
        o['_best_T_at_boundary_frac'] = float(np.mean(bnd))
        out[alpha] = o
        sd_m = o['_target_sd']['mean']
        ri = o['_reliability_identity']; rt = o['_reliability_after_T']
        print(f'--- [{tag}] alpha={alpha}  K={Ksub}  classes scored={np.mean(nsc):.0f}  '
              f'sd(q*_y)={sd_m:.4f} ---')
        for m in KEYS:
            mu = o[m]['mean']; lo = o[m]['ci_low']; hi = o[m]['ci_high']
            print(f'  R2 {m:14s} {mu:+8.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')
        print(f'  reliability of q_y: identity {ri["mean"]:+.3f} '
              f'[{ri["ci_low"]:+.3f},{ri["ci_high"]:+.3f}]'
              f'   worst over T {rt["mean"]:+.3f} '
              f'[{rt["ci_low"]:+.3f},{rt["ci_high"]:+.3f}]')
        if np.mean(bnd) > 0:
            print(f'  WARNING: temperature search hit the grid edge in '
                  f'{100*np.mean(bnd):.0f}% of splits -> widen Ts before trusting it')
    return out, Ksub, len(lab)

primary, K_prim, n_prim = run_explain(ABUNDANT, 'PRIMARY abundant')


## 5. SECONDARY - the same measurement on ALL classes

`min_eval` already restricts scoring to classes with enough EVAL samples, so this view is
well-posed rather than degenerate. Never merged with the primary.


In [ ]:
secondary = None
if RUN_UNRESTRICTED:
    secondary, K_sec, n_sec = run_explain(np.where(counts > 0)[0], 'SECONDARY all',
                                          n_splits=N_SPLITS_SECONDARY)
    print()
    print(f'(secondary used {N_SPLITS_SECONDARY} splits, so its CIs are wider by design)')
else:
    print('skipped')


## 6. SECONDARY - deployment COST (retired criterion, kept as a cost report)

**This table must not be used to rank mechanisms** (Amendment 6). It answers a different and
still-real question: what does class-conditional coverage COST at this calibration budget?


In [ ]:
cost = None
if RUN_COST_REPORT:
    lg2, lab2, Ksub2, _ = restrict_to_classes(logits, labels, ABUNDANT)
    cost = {}
    for alpha in ALPHAS:
        acc = {m: [] for m in ['temperature'] + MECHS}; gsize = []
        rng = np.random.default_rng(SEED)
        for _ in range(N_SPLITS_COST):
            idx = rng.permutation(len(lab2)); cal, ev = idx[:len(lab2)//2], idx[len(lab2)//2:]
            r = dc.phase0_cc_decomposition(lg2, lab2, Ksub2, alpha, cal, ev,
                                           bin_grid=BIN_GRID, estimator='empirical')
            gsize.append(r['global']['avg_set_size'])
            for m in acc: acc[m].append(r[m]['gap_vs_global'])
        cost[alpha] = {m: mean_ci(np.array(v, float)) for m, v in acc.items()}
        cost[alpha]['_global_size'] = mean_ci(np.array(gsize, float))
        print(f'--- [COST, not a ranking] alpha={alpha}  '
              f'global size {np.mean(gsize):.2f} ---')
        for m in acc:
            v = cost[alpha][m]
            print(f'  {m:14s} size gap {v["mean"]:+9.3f}')
else:
    print('skipped')


## 7. Gate verdict on the PRIMARY analysis (Sec 5, Amendment 6 criterion)


In [ ]:
verdicts = {}
for alpha in ALPHAS:
    r = primary[alpha]
    cls = r['class']
    rivals = {m: r[m] for m in MECHS if m != 'class'}
    best = max(rivals, key=lambda k: rivals[k]['mean'])
    ri, rt = r['_reliability_identity'], r['_reliability_after_T']
    exists   = bool(ri['ci_low'] > 0.30)
    survives = bool(rt['ci_low'] > 0.30)   # WORST temperature, not just the best-efficiency one
    beats    = bool(cls['ci_low'] > rivals[best]['ci_high'])
    verdicts[str(alpha)] = {
        'class_r2': cls['mean'], 'class_ci': [cls['ci_low'], cls['ci_high']],
        'best_rival': best, 'rival_r2': rivals[best]['mean'],
        'rival_ci': [rivals[best]['ci_low'], rivals[best]['ci_high']],
        'target_sd': r['_target_sd']['mean'],
        'reliability_identity': ri['mean'], 'reliability_after_T': rt['mean'],
        'structure_exists': exists, 'survives_temperature': survives,
        'class_beats_rivals': beats,
        'pass': bool(exists and survives and beats)}
    v = verdicts[str(alpha)]
    print(f'alpha={alpha}:')
    print(f'   structure exists above noise   r={ri["mean"]:+.3f} -> {exists}')
    print(f'   survives EVERY temperature     r={rt["mean"]:+.3f} -> {survives}')
    print(f'   class R2 {cls["mean"]:+.3f} beats {best} '
          f'{rivals[best]["mean"]:+.3f} -> {beats}')
    print('   => ' + ('PASS' if v['pass'] else 'FAIL'))

overall = 'PASS' if all(v['pass'] for v in verdicts.values()) else 'FAIL'
print()
print('PHASE 0 GATE (primary, abundant classes):', overall)
if overall != 'PASS':
    print('Sec 5: if the class mechanism does not dominate, the hypothesis that the')
    print('structure lives at the class level does not hold on real data -> report and STOP.')
print('NOTE: this verdict is readable only because the metric was first validated 4/4 on')
print('planted worlds (pcc/tests/test_phase0_explain.py); the retired set-size criterion')
print('scored 0/1 on the planted-class world.')


## 7. Write report


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    if isinstance(o, np.ndarray): return None
    return o

report = write_report('pcc/reports', f'03_phase0_decomposition_{DATASET}',
    hypothesis='on REAL logits, a per-CLASS offset closes a substantially larger efficiency gap '
               'than a global temperature or a per-sample energy-indexed offset',
    pass_criteria='class gap > best rival AND non-overlapping 95% CIs at every alpha, on the '
                  'PRIMARY abundant-class analysis (Sec 5 fits the per-class offset on abundant '
                  'data, not a realistic budget). Metric = avg set size at worst-class coverage '
                  '>= 1-alpha with out-of-sample per-group estimation (Amendment 3). The '
                  'unrestricted run is SECONDARY and never merged with the primary.',
    config=dict(dataset=DATASET, backbone=BACKBONE, cal_split=CAL_SPLIT,
                alphas=list(ALPHAS), min_cal_per_class=MIN_CAL_PER_CLASS,
                n_splits=N_SPLITS, n_splits_secondary=N_SPLITS_SECONDARY,
                bin_grid=list(BIN_GRID),
                n_abundant_classes=int(len(ABUNDANT)), n_classes_total=int(K),
                scores_source=prov.get('scores_source'),
                under_gate_exception=bool(prov.get('under_gate_exception')),
                amendments=['protocol_amendments.md#amendment-3']),
    seed=SEED,
    results={'primary_explain': clean(primary), 'verdicts': verdicts,
             'secondary_cost_not_a_ranking': clean(cost),
             'secondary_all_classes': clean(secondary),
             'cal_counts_percentiles': {f'p{q}': int(np.percentile(nz,q))
                                        for q in (0,25,50,75,90,100)}},
    conclusion=f'{overall} (primary: {len(ABUNDANT)} abundant classes of {K})',
    started_at=time.time())
print('report:', report)
print()
print('REMINDER: scores are OURS (gate exception) - cite reports/phase0_checkpoint_gate.md.')
